In [0]:
# Imports
from pyspark.sql.functions import (
    col, when, avg,
    count as spark_count,
    abs as spark_abs
)
from pyspark.sql.functions import round as spark_round


In [0]:
# Pipeline Test Suite
# EconMate data quality assertions
# Run this notebook after every full pipeline execution.
# ALL tests must pass before results are considered production-ready.
# Inspired by dbt's test philosophy: data without contracts is a guess.

print("ECONMATE — DATA QUALITY TEST SUITE")

passed = 0
failed = 0
failures = []

def run_test(test_name, condition, failure_message):
    global passed, failed
    if condition:
        print(f"  PASS: {test_name}")
        passed += 1
    else:
        print(f"  FAIL: {test_name} — {failure_message}")
        failed += 1
        failures.append(f"{test_name}: {failure_message}")

In [0]:
# Bronze row count tests
print("BRONZE LAYER")

df_gdp  = spark.table("workspace.default.bronze_world_bank_gdp")
df_clim = spark.table("workspace.default.bronze_open_meteo_climate")
df_food = spark.table("workspace.default.bronze_world_bank_food")
df_pop  = spark.table("workspace.default.bronze_world_bank_population")

run_test(
    "Bronze GDP row count",
    df_gdp.count() >= 200,
    f"Expected ≥200 rows, got {df_gdp.count()}"
)
run_test(
    "Bronze climate row count",
    df_clim.count() >= 70000,
    f"Expected ≥70,000 rows, got {df_clim.count()}"
)
run_test(
    "Bronze food row count",
    df_food.count() >= 300,
    f"Expected ≥300 rows, got {df_food.count()}"
)
run_test(
    "Bronze population row count",
    df_pop.count() >= 400,
    f"Expected ≥400 rows, got {df_pop.count()}"
)

# Schema tests — critical columns must exist
run_test(
    "Bronze GDP has country_code column",
    "country_code" in df_gdp.columns,
    "country_code column missing from bronze_world_bank_gdp"
)
run_test(
    "Bronze GDP has no null country_code",
    df_gdp.filter(col("country_code").isNull()).count() == 0,
    "Null country_codes found in bronze_world_bank_gdp"
)
run_test(
    "Bronze GDP has no null year",
    df_gdp.filter(col("year").isNull()).count() == 0,
    "Null years found in bronze_world_bank_gdp"
)
run_test(
    "Bronze climate covers all 15 countries",
    df_clim.select("country_code").distinct().count() == 15,
    f"Expected 15 countries, got {df_clim.select('country_code').distinct().count()}"
)
run_test(
    "Bronze GDP value range realistic",
    df_gdp.filter((col("value") < 0) | (col("value") > 100000)).count() == 0,
    "GDP values outside realistic range (0–100,000 USD)"
)
run_test(
    "Bronze climate precipitation non-negative",
    df_clim.filter(col("precipitation_mm") < 0).count() == 0,
    "Negative precipitation values found"
)

In [0]:
# Silver tests
print("SILVER LAYER")

df_dim  = spark.table("workspace.default.dim_country")
df_sgdp = spark.table("workspace.default.silver_fact_gdp")
df_sclim= spark.table("workspace.default.silver_fact_climate")
df_sfood= spark.table("workspace.default.silver_fact_food")
df_spop = spark.table("workspace.default.silver_fact_population")

run_test(
    "dim_country has exactly 15 rows",
    df_dim.count() == 15,
    f"Expected 15, got {df_dim.count()}"
)
run_test(
    "dim_country has no null country_code",
    df_dim.filter(col("country_code").isNull()).count() == 0,
    "Null country_code in dim_country"
)
run_test(
    "Silver GDP no duplicates on country+year",
    df_sgdp.count() == df_sgdp.select("country_code","year").distinct().count(),
    "Duplicate country+year combinations in silver_fact_gdp"
)
run_test(
    "Silver GDP gdp_per_capita_usd not null",
    df_sgdp.filter(col("gdp_per_capita_usd").isNull()).count() == 0,
    "Null GDP values in silver_fact_gdp"
)
run_test(
    "Silver GDP values rounded to 2dp",
    df_sgdp.filter(
        spark_round(col("gdp_per_capita_usd"), 2) != col("gdp_per_capita_usd")
    ).count() == 0,
    "GDP values not rounded to 2 decimal places"
)
run_test(
    "Silver climate covers all 15 countries",
    df_sclim.select("country_code").distinct().count() == 15,
    f"Expected 15 countries in silver_fact_climate"
)
run_test(
    "Silver climate no duplicate country+year+month",
    df_sclim.count() == df_sclim.select("country_code","year","month").distinct().count(),
    "Duplicate country+year+month in silver_fact_climate"
)
run_test(
    "Silver food undernourishment has 0 nulls",
    df_sfood.filter(col("undernourishment_pct").isNull()).count() == 0,
    "Null undernourishment values in silver_fact_food"
)
run_test(
    "Silver food values in realistic range (0-100%)",
    df_sfood.filter(
        (col("undernourishment_pct") < 0) | (col("undernourishment_pct") > 100)
    ).count() == 0,
    "Undernourishment values outside 0-100% range"
)
run_test(
    "Silver population total_population positive",
    df_spop.filter(col("total_population") <= 0).count() == 0,
    "Non-positive population values in silver_fact_population"
)
run_test(
    "Silver population urban_pct in range (0-100%)",
    df_spop.filter(
        (col("urban_population_pct") < 0) | (col("urban_population_pct") > 100)
    ).count() == 0,
    "Urban population % outside 0-100% range"
)

In [0]:
# Gold tests
print("GOLD LAYER")

df_gold = spark.table("workspace.default.gold_vulnerability_scores")

run_test(
    "Gold covers all 15 countries",
    df_gold.select("country_code").distinct().count() == 15,
    f"Expected 15 countries, got {df_gold.select('country_code').distinct().count()}"
)
run_test(
    "Gold vulnerability_index in range (0-100)",
    df_gold.filter(
        (col("vulnerability_index") < 0) | (col("vulnerability_index") > 100)
    ).count() == 0,
    "Vulnerability index values outside 0-100 range"
)
run_test(
    "Gold has no null vulnerability_index",
    df_gold.filter(col("vulnerability_index").isNull()).count() == 0,
    "Null vulnerability index values in gold table"
)
run_test(
    "Gold alert_level values are valid",
    df_gold.filter(
        ~col("alert_level").isin(["HIGH RISK", "ELEVATED", "MONITOR", "STABLE"])
    ).count() == 0,
    "Invalid alert_level values found"
)
run_test(
    "Gold trend values are valid",
    df_gold.filter(
        ~col("trend").isin(["WORSENING", "IMPROVING", "STABLE", "BASELINE"])
    ).count() == 0,
    "Invalid trend values found"
)
run_test(
    "Gold no duplicate country+year",
    df_gold.count() == df_gold.select("country_code","year").distinct().count(),
    "Duplicate country+year in gold_vulnerability_scores"
)
run_test(
    "Gold domain scores sum approximately to vulnerability index",
    df_gold.filter(
        spark_abs(
            (col("economic_score")*0.25 + col("climate_score")*0.25 +
             col("food_score")*0.35 + col("population_score")*0.15)*100
            - col("vulnerability_index")
        ) > 0.1
    ).count() == 0,
    "Vulnerability index does not match weighted domain scores"
)
run_test(
    "Gold South Africa has lowest average score",
    df_gold.groupBy("country_code").agg(avg("vulnerability_index").alias("avg_score"))
           .orderBy("avg_score").first()["country_code"] == "ZA",
    "South Africa is not the lowest-scoring country — check normalisation"
)

In [0]:
# Final test summary
print(f"TEST RESULTS: {passed} passed, {failed} failed")

if failed > 0:
    print("\nFAILURES:")
    for f in failures:
        print(f" {f}")
    raise Exception(f"Pipeline test suite FAILED: {failed} test(s) did not pass. Fix before proceeding.")
else:
    print("ALL TESTS PASSED.")